# **Import Libraries**

In [1]:
import joblib
import pandas as pd
import numpy as np

# **Load Model**

In [2]:
# load clustering model
cluster_model = joblib.load('cluster_model.pkl')

# load classification model
classification_model = joblib.load('classification_model.pkl')

# load threshold
threshold = joblib.load('threshold.pkl')

# **Decision**

In [3]:
def decision(prob):
    if prob < 0.20:
        return "APPROVED"
    elif prob < 0.40:
        return "REVIEWED"
    else:
        return "REJECTED"

# **Add Features**

In [4]:
def prepare_input(data_input):
    df = pd.DataFrame([data_input])
    
    df['spend_to_income'] = df['purchase_amount'] / (df['monthly_income'] + 1)
    df['income_per_installment'] = df['monthly_income'] / (df['bnpl_installments'] + 1)
    df['delay_ratio'] = df['repayment_delay_days'] / (df['bnpl_installments'] + 1)
    
    return df

# **Clustering**

In [5]:
def clustering(df):
    num_cols_cluster = [
        'age', 'monthly_income', 'credit_score',
        'repayment_delay_days', 'missed_payments', 
        'app_usage_frequency'
    ]
    
    df['cluster'] = cluster_model.predict(df[num_cols_cluster])
    
    cluster_map = {
        0: "Low Risk",
        1: "Potential Risk",
        2: "High Risk"
    }
    
    df['cluster_label'] = df['cluster'].map(cluster_map)
    
    return df

# **Prediction**

In [6]:
def predict(df):
    # drop unused + leakage columns
    drop_cols = [
        'product_category', 'location', 'transaction_date',
        'risk_score', 'default_flag'
    ]
    
    df_model = df.drop(columns=drop_cols, errors='ignore')
    
    y_proba = classification_model.predict_proba(df_model)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    
    prob = y_proba[0]
    
    label_map = {
        0: "Paid on Time",
        1: "Defaulted"
    }
    
    prediction_label = label_map[y_pred[0]]
    decision_result = decision(prob)
    
    return prob, prediction_label, decision_result

# **Credit Limit**

In [ ]:
def calculate_credit_limit(df, prob, decision):
    
    income = df['monthly_income'].iloc[0]
    
    # STEP 1: Affordability
    PTI = 0.3
    monthly_capacity = income * PTI
    
    # STEP 2: Cluster-based tenor 
    cluster = df['cluster'].iloc[0]
    requested_tenor = df['bnpl_installments'].iloc[0]
    if cluster == 0:        # Low Risk
        max_tenor = 12
    elif cluster == 1:      # Potential Risk
        max_tenor = 9
    else:                   # High Risk
        max_tenor = 6

    tenor = max(1, min(requested_tenor, max_tenor))

    # STEP 3: Base limit
    # Repayment capacity
    base_limit = monthly_capacity * tenor
    
    # STEP 4: Risk Factor (probability-based)
    # the smaller the probability, the larger the limit
    # minimum limit = 0.3 (if not given, too small)
    risk_factor = max(0.3, 1 - prob)
    
    # Step 5: Cluster Factor
    # cluster 0 = limit + 20%
    # cluster 1 = standard
    # cluster 2 = 50% limit
    if cluster == 0:
        cluster_factor = 1.2
    elif cluster == 1:
        cluster_factor = 1.0
    else:
        cluster_factor = 0.5
    
    # STEP 6: Decision Factor
    if decision == "APPROVE":
        decision_factor = 1.0
    elif decision == "REVIEW":
        decision_factor = 0.5
    else:
        decision_factor = 0.0
    
    credit_limit = base_limit * risk_factor * cluster_factor * decision_factor
    
    return max(0, round(credit_limit, 2))

# **Inference**

In [8]:
user1 = {
    'age': 30,
    'employment_type': 'Unemployed',
    'monthly_income': 450,  
    'credit_score': 50,
    'purchase_amount': 150,
    'product_category': 'Electronics',
    'bnpl_installments': 3,
    'repayment_delay_days': 19,
    'missed_payments': 10,
    'app_usage_frequency': 5,
    'location': 'Jakarta',
    'transaction_date': '2024-01-01',
    'debt_to_income_ratio': 15.9,
    'risk_score': 18.7,
    'customer_segment': 'High Risk'
}

In [9]:
user2 = {
    'age': 30,
    'employment_type': 'Salaried',
    'monthly_income': 60000,  
    'credit_score': 750,
    'purchase_amount': 5000,
    'product_category': 'Electronics',
    'bnpl_installments': 3,
    'repayment_delay_days': 0,
    'missed_payments': 0,
    'app_usage_frequency': 5,
    'location': 'Tangerang',
    'transaction_date': '2026-05-03',
    'debt_to_income_ratio': 0,
    'risk_score': 10,
    'customer_segment': 'Medium Risk'
}

In [10]:
user3 = {
    'age': 20,
    'employment_type': 'Salaried',
    'monthly_income': 20000,  
    'credit_score': 400,
    'purchase_amount': 37000,
    'product_category': 'Electronics',
    'bnpl_installments': 4,
    'repayment_delay_days': 0,
    'missed_payments': 0,
    'app_usage_frequency': 5,
    'location': 'Tangerang',
    'transaction_date': '2026-05-03',
    'debt_to_income_ratio': 0.2,
    'risk_score': 200,
    'customer_segment': 'Medium Risk'
}

# **Predict the Data**

In [11]:
df_user1 = prepare_input(user1)
df_user1 = clustering(df_user1)

prob, pred_label, decision_result = predict(df_user1)
limit = calculate_credit_limit(df_user1, prob, decision_result)

print("MODEL INFERENCE RESULT")
print("----------------------")
print(f"Risk Segment: {df_user1['cluster_label'].iloc[0]}")
print(f"Probability of Default: {prob:.2%}")
print(f"Prediction: {pred_label}")
print(f"Decision: {decision_result}")

if decision_result != "REJECT":
    print(f"Recommended Credit Limit: $ {limit:,.2f}")
else:
    print("Recommended Credit Limit: Not Available (Rejected)")

MODEL INFERENCE RESULT
----------------------
Risk Segment: High Risk
Probability of Default: 77.59%
Prediction: Defaulted
Decision: REJECTED
Recommended Credit Limit: $ 0.00


The inference result shows that the customer is classified as **High Risk** with a high probability of default (79.58%). Based on this prediction, the model labels the customer as *Defaulted* and assigns a **REJECT** decision. Consequently, no credit limit is provided, which is consistent with the customer’s profile characterized by low income, very low credit score, and poor repayment behavior.

In [12]:
df_user2 = prepare_input(user2)
df_user2 = clustering(df_user2)

prob, pred_label, decision_result = predict(df_user2)
limit = calculate_credit_limit(df_user2, prob, decision_result)

print("MODEL INFERENCE RESULT")
print("----------------------")
print(f"Risk Segment: {df_user2['cluster_label'].iloc[0]}")
print(f"Probability of Default: {prob:.2%}")
print(f"Prediction: {pred_label}")
print(f"Decision: {decision_result}")

if decision_result != "REJECT":
    print(f"Recommended Credit Limit: $ {limit:,.2f}")
else:
    print("Recommended Credit Limit: Not Available (Rejected)")

MODEL INFERENCE RESULT
----------------------
Risk Segment: Low Risk
Probability of Default: 1.39%
Prediction: Paid on Time
Decision: APPROVED
Recommended Credit Limit: $ 0.00


The inference result shows that the customer is classified as **Low Risk** with a very low probability of default (1.19%). Based on this prediction, the model labels the customer as *Paid on Time* and assigns an **APPROVE** decision. Consequently, a credit limit is provided, which is consistent with the customer’s profile characterized by high income, a strong credit score, and good repayment behavior.

In [13]:
df_user3 = prepare_input(user3)
df_user3 = clustering(df_user3)

prob, pred_label, decision_result = predict(df_user3)
limit = calculate_credit_limit(df_user3, prob, decision_result)

print("MODEL INFERENCE RESULT")
print("----------------------")
print(f"Risk Segment: {df_user3['cluster_label'].iloc[0]}")
print(f"Probability of Default: {prob:.2%}")
print(f"Prediction: {pred_label}")
print(f"Decision: {decision_result}")

if decision_result != "REJECT":
    print(f"Recommended Credit Limit: $ {limit:,.2f}")
else:
    print("Recommended Credit Limit: Not Available (Rejected)")

MODEL INFERENCE RESULT
----------------------
Risk Segment: Potential Risk
Probability of Default: 26.97%
Prediction: Defaulted
Decision: REVIEWED
Recommended Credit Limit: $ 0.00


The inference result shows that the customer is classified as **Potential Risk** with a moderate probability of default (28.59%). Based on this prediction, the model labels the customer as *Defaulted* and assigns a **REVIEW** decision. Consequently, a reduced credit limit is provided, which reflects a cautious approach given the customer’s moderate income, relatively low credit score, and high purchase amount compared to their financial capacity.